# QuantumCircuit.jl Phase 3A Fixed-Qubit Coupler Resonator Dynamics

**Audience**
- Julia users who already know the static and sweep workflows and want a mixed fixed-frequency qubit example for Phase 3A dynamics.

**Prerequisites**
- Basic Julia syntax.
- Familiarity with `CompositeSystem`, `spectrum`, and the Phase 2 sweep helpers.

**What you will learn**
- How to build a `Transmon - TunableCoupler - Resonator` chain with the current public API.
- How to choose a static coupler-flux operating point before running dynamics.
- How to drive the fixed qubit locally with `SubsystemDrive` and inspect transferred response on `c1` and `r1`.
- How to cross-check the population plots with `ObservableSpec` number traces.


## Outline

1. Activate the local package environment.
2. Build a `qf-c1-r1` chain and inspect its low-lying spectrum and coupler-flux sweep.
3. Run an undriven baseline and a driven evolution at one operating point.
4. Plot the directly driven qubit response and the transferred chain response.
5. Compare resonator transfer at two static coupler-flux values.
6. Review the current Phase 3A limits and try one exercise.


In [2]:
using Pkg

function find_repo_root(start::AbstractString)
    candidates = [
        normpath(start),
        normpath(joinpath(start, "..")),
        normpath(joinpath(start, "..", "..")),
    ]

    for candidate in unique(candidates)
        project_toml = joinpath(candidate, "Project.toml")
        if isfile(project_toml)
            content = read(project_toml, String)
            occursin("QuantumCircuit", content) && return candidate
        end
    end

    error("Could not find the QuantumCircuit.jl project root. Start Jupyter from the repository or open the notebook from inside it.")
end

project_root = find_repo_root(pwd())
Pkg.activate(project_root)
Pkg.instantiate()

using QuantumCircuit
using UnicodePlots


  Activating project at `~/Research/20_Projects/QuantumCircuit.jl`


## Step 1 - Build the mixed system and inspect the static operating point

This notebook uses a fixed-frequency qubit, a tunable coupler, and a resonator in a short chain. The coupler flux stays static during each run in Phase 3A, so we first inspect the spectrum and a small flux sweep before choosing one operating point for dynamics.


## Step 1A - Hamiltonian used in this notebook

The fixed qubit `qf` uses the current Duffing-style approximation,

$$
\omega_{\mathrm{qf}} = \sqrt{8 E_{J,\mathrm{qf}} E_{C,\mathrm{qf}}} - E_{C,\mathrm{qf}}, \qquad \alpha_{\mathrm{qf}} = -E_{C,\mathrm{qf}},
$$

$$
\hat H_{\mathrm{qf}} = \omega_{\mathrm{qf}} \, \hat n_{\mathrm{qf}} + \frac{\alpha_{\mathrm{qf}}}{2} \, \hat n_{\mathrm{qf}} (\hat n_{\mathrm{qf}} - I).
$$

The tunable coupler `c1` uses the same Duffing form, but with flux-dependent Josephson energy,

$$
E_{J,\mathrm{c1}}^{\mathrm{eff}}(\Phi) = E_{J,\mathrm{c1}}^{\max} \sqrt{\cos^2(\pi \Phi) + d_{\mathrm{c1}}^2 \sin^2(\pi \Phi)},
$$

$$
\omega_{\mathrm{c1}}(\Phi) = \sqrt{8 E_{J,\mathrm{c1}}^{\mathrm{eff}}(\Phi) E_{C,\mathrm{c1}}} - E_{C,\mathrm{c1}}, \qquad \alpha_{\mathrm{c1}} = -E_{C,\mathrm{c1}},
$$

$$
\hat H_{\mathrm{c1}}(\Phi) = \omega_{\mathrm{c1}}(\Phi) \, \hat n_{\mathrm{c1}} + \frac{\alpha_{\mathrm{c1}}}{2} \, \hat n_{\mathrm{c1}} (\hat n_{\mathrm{c1}} - I).
$$

The resonator contributes the harmonic term

$$
\hat H_{\mathrm{r1}} = \omega_{\mathrm{r1}} \, \hat a_{\mathrm{r1}}^\dagger \hat a_{\mathrm{r1}}.
$$

The current `CapacitiveCoupling` implementation gives

$$
\hat H_{\mathrm{int}} = g_{\mathrm{qf},\mathrm{c1}} \left( \hat a_{\mathrm{qf}}^\dagger \hat a_{\mathrm{c1}} + \hat a_{\mathrm{qf}} \hat a_{\mathrm{c1}}^\dagger \right) + g_{\mathrm{c1},\mathrm{r1}} \left( \hat a_{\mathrm{c1}}^\dagger \hat a_{\mathrm{r1}} + \hat a_{\mathrm{c1}} \hat a_{\mathrm{r1}}^\dagger \right),
$$

and the local `:x` drive on `qf` adds

$$
\hat H_d(t) = \Omega \cos(\omega_d t) \left( \hat a_{\mathrm{qf}} + \hat a_{\mathrm{qf}}^\dagger \right).
$$

So the driven problem below is

$$
\hat H(t; \Phi) = \hat H_{\mathrm{qf}} + \hat H_{\mathrm{c1}}(\Phi) + \hat H_{\mathrm{r1}} + \hat H_{\mathrm{int}} + \hat H_d(t).
$$

As elsewhere in the package, Hamiltonian entries, drive frequencies, coupling strengths, and evolution times all use the same frequency-unit convention with $\hbar = 1$.


In [3]:
function build_qf_c1_r1(flux::Float64)
    qf = Transmon(:qf; EJ = 19.0, EC = 0.24, ncut = 3)
    c1 = TunableCoupler(:c1; EJmax = 15.0, EC = 0.30, flux = flux, asymmetry = 0.10, ncut = 3)
    r1 = Resonator(:r1; ω = 6.7, dim = 3)

    return CompositeSystem(
        qf,
        c1,
        r1,
        CapacitiveCoupling(:qf, :c1; g = 0.12),
        CapacitiveCoupling(:c1, :r1; g = 0.12),
    )
end

sys = build_qf_c1_r1(0.0)
spec = spectrum(sys; levels = 5)
flux_sweep = simulate_sweep(sys, SweepSpec(:c1, :flux, [0.0, 0.06, 0.12, 0.18]; levels = 5))
summary = sweep_summary(flux_sweep)

(; low_lying_energies = spec.energies, coupler_flux_values = summary.values, transition_01 = summary.transition_01)


(low_lying_energies = [0.0, 5.610669959297679, 5.87477846487212, 6.714419124046792, 11.006400167099654], coupler_flux_values = [0.0, 0.06, 0.12, 0.18], transition_01 = [5.610669959297679, 5.571382469379284, 5.43671005763907, 5.18582067783656])

## Step 2 - Run baseline and driven dynamics

We start in the product ground state and keep the coupler flux fixed during each run. The baseline evolution should remain quiet, while the driven run applies a local `x` drive only to the fixed qubit `qf`.


In [4]:
function run_qf_c1_r1(sys; Ω = 0.20, ωd = 5.8, tlist = collect(range(0.0, 16.0; length = 121)))
    ψ0 = basis_state(sys; qf = 0, c1 = 0, r1 = 0)
    drive = SubsystemDrive(
        :qf_x_drive,
        :qf,
        :x,
        (p, t) -> p.Ω * cos(p.ωd * t),
    )

    baseline = evolve(
        sys,
        ψ0,
        tlist;
        observables = [ObservableSpec(:nqf, :qf, :n), ObservableSpec(:nr1, :r1, :n)],
    )
    driven = evolve(
        sys,
        ψ0,
        tlist;
        drives = [drive],
        observables = [ObservableSpec(:nqf, :qf, :n), ObservableSpec(:nr1, :r1, :n)],
        params = (; Ω, ωd),
    )

    return baseline, driven
end

baseline, driven = run_qf_c1_r1(sys)

qf_baseline = population_trace(baseline, :qf, 1)
qf_population = population_trace(driven, :qf, 1)
c1_population = population_trace(driven, :c1, 1)
r1_population = population_trace(driven, :r1, 1)
qf_number = observable_trace(driven, :nqf)
r1_number = observable_trace(driven, :nr1)

(; max_qf_population = maximum(qf_population.values), max_c1_population = maximum(c1_population.values), max_r1_population = maximum(r1_population.values), qf_number_peak = maximum(real.(qf_number.values)), r1_number_peak = maximum(real.(r1_number.values)))


(max_qf_population = 0.5389636323601968, max_c1_population = 0.5304364937780423, max_r1_population = 0.010406124410279407, qf_number_peak = 0.7914598167926483, r1_number_peak = 0.010447041145402139)

## Step 3 - Compare the directly driven qubit against baseline

This first plot isolates the local qubit response. The undriven baseline stays flat, while the driven `qf` excited-state population grows under the local drive.


In [5]:
qf_plot = lineplot(
    qf_baseline.times,
    qf_baseline.values;
    name = "baseline",
    title = "qf excited-state population: baseline vs driven",
    xlabel = "time",
    ylabel = "P_qf(|1>)",
)
lineplot!(qf_plot, qf_population.times, qf_population.values; name = "driven")
qf_plot


                qf excited-state population: baseline vs driven         
                ┌────────────────────────────────────────┐         
              1 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│ baseline
                │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│ driven  
                │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│         
                │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣀⣀⣤⠤⠤⠤⠶⠒⠀⠀⠀⠀⠀⠀⠀⠀│         
                │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣀⡠⠤⠔⠒⠊⠉⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│         
                │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⣀⡤⠴⠒⠉⠉⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│         
                │⠀⠀⠀⠀⠀⢀⣠⠤⠔⠚⠉⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│         
   P_qf(|1>)    │⠶⠶⠶⠮⠭⠥⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤⠤│         
                │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│         
                │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│         
                │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│         
                │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│         
                │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀

## Step 4 - Plot transfer through the coupler into the resonator

Now compare all three local excited-state populations. `qf` is driven directly, `c1` carries the intermediate response, and `r1` picks up a smaller but still visible transferred signal at this operating point.


In [6]:
transfer_plot = lineplot(
    qf_population.times,
    qf_population.values;
    name = "qf",
    title = "Population transfer from qf through c1 into r1",
    xlabel = "time",
    ylabel = "P(|1>)",
)
lineplot!(transfer_plot, c1_population.times, c1_population.values; name = "c1")
lineplot!(transfer_plot, r1_population.times, r1_population.values; name = "r1")
transfer_plot


              Population transfer from qf through c1 into r1   
              ┌────────────────────────────────────────┐   
          0.6 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│ qf
              │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣀⣄⢤⢤⠀⠀⠀⠀⠀⠀⠀⠀│ c1
              │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣠⠶⠉⠉⢀⡔⠁⠀⠀⠀⠀⠀⠀⠀⠀│ r1
              │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢠⡰⠛⠀⠀⠀⡠⠋⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│   
              │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣠⠜⠁⠀⠀⠀⢀⠖⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│   
              │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣠⠎⠁⠀⠀⠀⠀⡠⠃⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│   
              │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣠⠞⠁⠀⠀⠀⠀⠀⡴⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│   
   P(|1>)     │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣠⠒⠁⠀⠀⠀⠀⠀⢠⠎⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│   
              │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢠⠖⠁⠀⠀⠀⠀⠀⠀⡠⠃⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│   
              │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⠴⠉⠀⠀⠀⠀⠀⠀⠀⡔⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│   
              │⠀⠀⠀⠀⠀⠀⠀⠀⠀⢰⠊⠀⠀⠀⠀⠀⠀⠀⣠⠋⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│   
              │⠀⠀⠀⠀⠀⠀⠀⢀⠼⠁⠀⠀⠀⠀⠀⠀⢀⡔⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│   
              │⠀⠀⠀⠀⠀⠀⡰⠃⠀⠀⠀⠀⠀⠀⢀⡤⠋⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│   
              │⠀⠀⠀⠀⡠⠎⠁⠀⠀⠀⠀⠀⣀⠔⠋⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│   
            0 │⣀⣀⣤⣋⣁⣀⣀⣀⣠⣤⣒⣉⣁⣀⣀⣀⣀⣀⣀⣀⣀

## Step 5 - Compare resonator response at two coupler-flux values

The coupler flux stays static within each run, but changing that operating point changes how much of the driven qubit response reaches the resonator. Here we compare a low-flux point with a more weakly transmitting high-flux point.


In [7]:
low_flux_sys = build_qf_c1_r1(0.0)
high_flux_sys = build_qf_c1_r1(0.18)

_, low_flux_driven = run_qf_c1_r1(low_flux_sys)
_, high_flux_driven = run_qf_c1_r1(high_flux_sys)

r1_flux_0 = population_trace(low_flux_driven, :r1, 1)
r1_flux_018 = population_trace(high_flux_driven, :r1, 1)

flux_plot = lineplot(
    r1_flux_0.times,
    r1_flux_0.values;
    name = "flux = 0.00",
    title = "Resonator response at two coupler flux points",
    xlabel = "time",
    ylabel = "P_r1(|1>)",
)
lineplot!(flux_plot, r1_flux_018.times, r1_flux_018.values; name = "flux = 0.18")
flux_plot


                  Resonator response at two coupler flux points            
                  ┌────────────────────────────────────────┐            
             0.02 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│ flux = 0.00
                  │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│ flux = 0.18
                  │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│            
                  │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│            
                  │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│            
                  │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│            
                  │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│            
   P_r1(|1>)      │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⡠⠒⠀⠀⠀⠀⠀⠀⠀⠀│            
                  │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡠⠔⠉⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│            
                  │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⡴⠋⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│            
                  │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⡤⠋⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│            
                  │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡠⠋⠀⠀⠀⠀⠀⠀

## Interpretation

- `qf` is the only directly driven subsystem in this example.
- `c1` shows a strong intermediate response, which is why the resonator trace does not remain flat.
- The resonator response is smaller than the qubit and coupler response, but it is still visible at the low-flux operating point.
- The `ObservableSpec(:nqf, :qf, :n)` and `ObservableSpec(:nr1, :r1, :n)` traces are useful secondary checks that the driven run excites both the qubit and the resonator.
- Raising the static coupler flux suppresses transferred resonator response even though the drive API itself is still purely subsystem-local.


## Current Limits

- This is still a closed-system example. There is no relaxation or dephasing in Phase 3A.
- The coupler flux is a static operating-point parameter here, not a time-dependent flux pulse.
- The local subsystem models are Duffing-style approximations, so this notebook should be read as an intuition-building workflow rather than a calibrated device model.


## Exercise

Increase both couplings from `0.12` to `0.15` and compare how much more of the driven response reaches the resonator.
The quickest check is to rerun the low-flux operating point with stronger couplings and overlay the `r1` trace.


In [8]:
stronger_chain = CompositeSystem(
    Transmon(:qf; EJ = 19.0, EC = 0.24, ncut = 3),
    TunableCoupler(:c1; EJmax = 15.0, EC = 0.30, flux = 0.0, asymmetry = 0.10, ncut = 3),
    Resonator(:r1; ω = 6.7, dim = 3),
    CapacitiveCoupling(:qf, :c1; g = 0.15),
    CapacitiveCoupling(:c1, :r1; g = 0.15),
)

_, stronger_driven = run_qf_c1_r1(stronger_chain)
stronger_r1 = population_trace(stronger_driven, :r1, 1)

exercise_plot = lineplot(
    r1_population.times,
    r1_population.values;
    name = "g = 0.12",
    title = "Exercise: stronger coupling increases resonator transfer",
    xlabel = "time",
    ylabel = "P_r1(|1>)",
)
lineplot!(exercise_plot, stronger_r1.times, stronger_r1.values; name = "g = 0.15")
exercise_plot


                  Exercise: stronger coupling increases resonator transfer         
                  ┌────────────────────────────────────────┐         
             0.02 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│ g = 0.12
                  │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡠⠀⠀⠀⠀⠀⠀⠀⠀│ g = 0.15
                  │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⡠⠊⠀⠀⠀⠀⠀⠀⠀⠀⠀│         
                  │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⠔⠋⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│         
                  │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡠⠋⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│         
                  │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡴⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│         
                  │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⠎⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│         
   P_r1(|1>)      │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⠎⠀⠀⠀⠀⠀⠀⢀⡠⠒⠀⠀⠀⠀⠀⠀⠀⠀│         
                  │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⠎⠀⠀⠀⠀⠀⡠⠔⠉⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│         
                  │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⠎⠀⠀⠀⢀⡴⠋⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│         
                  │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡰⠁⠀⠀⢀⡤⠋⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│         
                  │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣠⠞⠀⠀⠀⡠⠋⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│         
      